
# Extração de Características linguísticas

In [1]:
import os
import shutil
import gc
from pathlib import Path
from time import perf_counter
import math

import pandas as pd
import numpy as np
import spacy
import lftk
from tqdm import tqdm
from sklearn.tree import DecisionTreeClassifier

In [2]:
BASE_DIR = Path.cwd().parent 

FILE_PATH = BASE_DIR / "data/df_macrotopics.csv" 
OUTPUT_PATH = BASE_DIR / "data/lftk_output_features.csv"
LFTK_FILE = BASE_DIR / "data/lftk_output_features.csv"
TEMP_DIR = BASE_DIR / "data/temp_partitions"

COLUMN = "text"
TARGET_COLUMN = "macrotopico" 
BATCH_SIZE = 32   
CHUNK_SIZE = 1000

In [3]:
def stream_lftk_features(texts: list, nlp: spacy.Language, features: list) -> pd.DataFrame:
    """Extrai features em formato de fluxo (stream) para economizar memória RAM."""
    results = []
    pipes_to_disable = ["ner", "textcat", "textcat_multilabel", "entity_linker", "entity_ruler"]
    
    doc_generator = nlp.pipe(
        texts, 
        batch_size=BATCH_SIZE, 
        disable=pipes_to_disable, 
        n_process=-1 
    )
    
    for doc in tqdm(doc_generator, total=len(texts), desc="spaCy + LFTK"):
        extractor = lftk.Extractor(docs=doc)
        extractor.customize(stop_words=True, punctuations=True)   
        feats = extractor.extract(features=features)
        results.append(feats)
        
    return pd.DataFrame(results)

In [4]:
def calculate_gini_gain(df: pd.DataFrame, feature_cols: list, target_col: str) -> pd.DataFrame:
    """
    Calcula o Gini Gain e extrai o limiar de split de cada feature usando Stumps.
    Retorna um DataFrame formatado.
    """
    gains = []
    
    # Tratamento de NaNs
    df_clean = df.dropna(subset=feature_cols + [target_col])
    y = df_clean[target_col]
    
    for col in feature_cols:
        X = df_clean[[col]]
        tree = DecisionTreeClassifier(max_depth=1, criterion='gini', random_state=42)
        tree.fit(X, y)
        
        # Computa a impureza e o limiar (split) do nó raiz
        gini_gain = tree.tree_.compute_feature_importances(normalize=False)[0]
        split_val = tree.tree_.threshold[0]
        
        gains.append({
            'attribute': col,
            'split': split_val,
            'gini_gain': gini_gain
        })
        
    # Converte para DataFrame e ordena do maior para o menor ganho
    df_ranked = pd.DataFrame(gains).sort_values(by='gini_gain', ascending=False).reset_index(drop=True)
    return df_ranked

In [5]:
def filter_features_by_spearman(df: pd.DataFrame, ranked_df: pd.DataFrame, threshold: float = 0.7) -> list:
    """
    Filtra features redundantes por Spearman usando o ranking baseado em DataFrame.
    """
    selected_features = []
    features_to_eval = ranked_df['attribute'].tolist()
    
    # Matriz de correlação de Spearman
    corr_matrix = df[features_to_eval].corr(method='spearman').abs()
    
    while features_to_eval:
        top_feat = features_to_eval.pop(0)
        selected_features.append(top_feat)
        
        correlated = corr_matrix.index[(corr_matrix[top_feat] > threshold)].tolist()
        features_to_eval = [f for f in features_to_eval if f not in correlated and f != top_feat]
            
    return selected_features

In [20]:
df = pd.read_csv(FILE_PATH)
nlp = spacy.load("pt_core_news_lg")

all_features = lftk.search_features(return_format="list_key")
print(f"Total de features configuradas: {len(all_features)}")

# Processamento LFTK em Lotes
TEMP_DIR.mkdir(parents=True, exist_ok=True)
temp_files = []
total_chunks = math.ceil(len(df) / CHUNK_SIZE)

Total de features configuradas: 220


### Extração LFTK em lotes

In [ ]:
TEMP_DIR.mkdir(parents=True, exist_ok=True)
total_chunks = math.ceil(len(df) / CHUNK_SIZE)

print(f"Total de lotes previstos: {total_chunks}")


for i in range(total_chunks):
    temp_file = TEMP_DIR / f"part_{i:04d}.parquet"
    
    if temp_file.exists():
        print(f"\n[CACHE] Lote {i+1}/{total_chunks} já processado anteriormente. Pulando...")
        continue
        
    print(f"\n[PROCESSANDO] Lote {i+1}/{total_chunks}")
    start_idx = i * CHUNK_SIZE
    chunk_df = df.iloc[start_idx : start_idx + CHUNK_SIZE].reset_index(drop=True)
    chunk_texts = chunk_df[COLUMN].astype(str).tolist()

    start_time = perf_counter()
    
    df_feats = stream_lftk_features(chunk_texts, nlp, all_features)
    print(f"Lote finalizado em {perf_counter() - start_time:.2f}s")

    chunk_final = pd.concat([chunk_df, df_feats], axis=1)
    
    chunk_final.to_parquet(temp_file, engine='fastparquet')
    
    del chunk_df, chunk_texts, df_feats, chunk_final
    gc.collect()

print("\nConsolidando todos os lotes encontrados...")

arquivos_completos = sorted(list(TEMP_DIR.glob("part_*.parquet")))

if not arquivos_completos:
    raise FileNotFoundError("Nenhum arquivo de lote encontrado para consolidação.")

# Junta todos os lotes que foram processados 
df_final = pd.concat([pd.read_parquet(f, engine='fastparquet') for f in arquivos_completos], ignore_index=True)

gc.collect()
shutil.rmtree(TEMP_DIR, ignore_errors=True)

df_final.to_csv(OUTPUT_PATH, index=False)

Total de lotes previstos: 7

[CACHE] Lote 1/7 já processado anteriormente. Pulando...

[CACHE] Lote 2/7 já processado anteriormente. Pulando...

[CACHE] Lote 3/7 já processado anteriormente. Pulando...

[CACHE] Lote 4/7 já processado anteriormente. Pulando...

[CACHE] Lote 5/7 já processado anteriormente. Pulando...

[CACHE] Lote 6/7 já processado anteriormente. Pulando...

[CACHE] Lote 7/7 já processado anteriormente. Pulando...

Consolidando todos os lotes encontrados...


In [22]:
df_final.to_csv(OUTPUT_PATH)

In [6]:
print(f"[1/3] Carregando dados de: {FILE_PATH}")
df_macro = pd.read_parquet(FILE_PATH) if str(FILE_PATH).endswith('.parquet') else pd.read_csv(FILE_PATH)

print("[2/3] Aplicando limpeza de Regex, removendo nulos e duplicatas...")
len_orig_macro = len(df_macro)
df_macro = df_macro.dropna(subset=[COLUMN]).drop_duplicates(subset=[COLUMN])
print(f"    -> Registros originais : {len_orig_macro:,}")
print(f"    -> Registros limpos    : {len(df_macro):,}")

# --- 2. Carga e "Limpeza" das Features ---
print(f"[1/3] Carregando dados de: {LFTK_FILE}")
df_lftk = pd.read_parquet(LFTK_FILE) if str(LFTK_FILE).endswith('.parquet') else pd.read_csv(LFTK_FILE)

print("[2/3] Aplicando limpeza de Regex, removendo nulos e duplicatas...")
len_orig_lftk = len(df_lftk)
df_lftk = df_lftk.drop_duplicates()
print(f"    -> Registros originais : {len_orig_lftk:,}")
print(f"    -> Registros limpos    : {len(df_lftk):,}")

# --- 3. Merge e Filtros Finais ---
# Alinhando os indexadores para concatenar lado a lado corretamente
df_macro.reset_index(drop=True, inplace=True)
df_lftk.reset_index(drop=True, inplace=True)

df_final = pd.concat([df_macro, df_lftk], axis=1)

# Previne que colunas se dupliquem durante o concat (ex: se "text" vier de ambos)
df_final = df_final.loc[:, ~df_final.columns.duplicated()]
print(f"Shape após merge: {df_final.shape}")

df_final = df_final.dropna(subset=[TARGET_COLUMN])
print(f"Shape após filtro de target: {df_final.shape}")

# --- 4. Gini Gain ---
print("Calculando o gini gain de cada métrica...")
# Traz as features suportadas do LFTK pra evitar KeyError
all_features = lftk.search_features(return_format="list_key")
feature_cols = [col for col in all_features if col in df_final.columns]

ranked_df = calculate_gini_gain(df_final, feature_cols, TARGET_COLUMN)

# --- 5. Spearman e Interseção (A Mágica da Visualização) ---
print("Selecionando o top10 métricas mais representativas...")
final_selected_features = filter_features_by_spearman(df_final, ranked_df, threshold=0.7)

top_10_before = ranked_df['attribute'].head(10).tolist()
top_10_after = final_selected_features[:10]

# Identifica o que saiu e o que entrou
removidos = [f for f in top_10_before if f not in top_10_after]
substituidos = [f for f in top_10_after if f not in top_10_before]

# Print do Top 10 Antes
print("\nTop 10 antes da correlação de Spearman:")
for i, feat in enumerate(top_10_before, 1):
    print(f"    {i:2}. {feat}")
    
# Print do Top 10 Depois
print(f"\nTop 10 após correlação de Spearman (threshold=0.7):")
for i, feat in enumerate(top_10_after, 1):
    novo_tag = "  <-- novo" if feat in substituidos else ""
    print(f"    {i:2}. {feat}{novo_tag}")
    
# Resumo das modificações
print(f"\nRemovidos por correlação: {removidos}")
print(f"Substituídos por:         {substituidos}")

# Renderização da Tabela Final
df_top10_print = ranked_df.set_index('attribute').loc[top_10_after].reset_index()
df_top10_print['split'] = df_top10_print['split'].apply(lambda x: f"{x:.3f}")
df_top10_print['gini_gain'] = df_top10_print['gini_gain'].apply(lambda x: f"{x:.6f}")

print("\n")
print(df_top10_print[['attribute', 'split', 'gini_gain']].to_string(index=True))

# --- 6. Exportação ---
cols_to_keep = list(df_macro.columns) + final_selected_features
# Garante que só puxaremos colunas que realmente existem no df_final
cols_to_keep = [c for c in cols_to_keep if c in df_final.columns]

df_filtered = df_final[cols_to_keep]
df_filtered.to_csv("gini.csv", index=False)

[1/3] Carregando dados de: c:\Users\Nathan\Documents\lei_felca\data\df_macrotopics.csv
[2/3] Aplicando limpeza de Regex, removendo nulos e duplicatas...
    -> Registros originais : 6,432
    -> Registros limpos    : 6,432
[1/3] Carregando dados de: c:\Users\Nathan\Documents\lei_felca\data\lftk_output_features.csv
[2/3] Aplicando limpeza de Regex, removendo nulos e duplicatas...
    -> Registros originais : 6,432
    -> Registros limpos    : 6,432
Shape após merge: (6432, 242)
Shape após filtro de target: (6432, 242)
Calculando o gini gain de cada métrica...
Selecionando o top10 métricas mais representativas...

Top 10 antes da correlação de Spearman:
     1. cole
     2. a_char_pw
     3. smog
     4. a_syll_pw
     5. fkre
     6. auto
     7. fkgl
     8. fogi
     9. a_adp_ps
    10. a_syll_ps

Top 10 após correlação de Spearman (threshold=0.7):
     1. cole
     2. a_adp_pw  <-- novo
     3. a_cconj_ps  <-- novo
     4. t_syll3  <-- novo
     5. a_punct_pw  <-- novo
     6. a_subt